# 自动化测试与类型检查

学习目标：用项目级类型检查和 Node.js 运行测试分别验证静态契约、同步结果及异步失败。

前置知识：tsconfig、函数返回类型、Promise、异常与模块导入。

适用版本与条件：TypeScript 7.0.2、Node.js 24.11.0；使用 ES 模块，开启 strict。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/typescript。

配套脚本：位于 scripts/23-testing/。

1. [calculator.ts](scripts/23-testing/calculator.ts)：待测同步与异步接口。
2. [calculator.test.ts](scripts/23-testing/calculator.test.ts)：Node.js 的行为测试。
3. [type-tests.ts](scripts/23-testing/type-tests.ts)：正常类型关系与预期类型错误。
4. [type-errors.ts](scripts/23-testing/type-errors.ts)：不被抑制的原始诊断。
5. [tsconfig.json](scripts/23-testing/tsconfig.json)：正常项目；反例使用 tsconfig.errors.json。

## 1 确定检查范围与测试入口

tsc -p 读取整份项目配置，不能只对一个片段检查后就认定模块边界也通过。正常配置包含实现、行为测试、类型测试；错误配置只以 type-errors.ts 为入口，再加载其导入的实现。

类型测试里的错误调用只用于检查，不是要执行的业务输入。run:23 仅执行生成的 calculator.test.js，避免误执行 type-tests.js 中的反例。

以下片段来自 tsconfig.json。

```json
{
  "compilerOptions": {
    "target": "ES2025",
    "module": "NodeNext",
    "moduleResolution": "NodeNext",
    "strict": true,
    "lib": [
      "ES2025"
    ],
    "types": [
      "node"
    ],
    "rootDir": ".",
    "outDir": "./.build",
    "noEmitOnError": true
  },
  "files": [
    "calculator.ts",
    "calculator.test.ts",
    "type-tests.ts"
  ]
}
```

Step 1：检查本章正常项目。

```bash
npm run check:23
# 无类型诊断。
```

Step 2：生成当前源码的 JavaScript。

```bash
npm run build:23
# 类型错误时不生成新输出。
```

Step 3：执行本章运行入口。

```bash
npm run run:23
# node --test 执行三项行为测试，fail 为 0。
```

## 2 为正常和失败行为建立最小接口

number 参数能阻止字符串调用，却不限制除数非零。除零是本例的运行时规则，divide 明确抛 RangeError。异步包装返回 Promise&lt;number&gt;；同步抛出的异常会成为该 async 函数返回 Promise 的拒绝。

以下片段来自 calculator.ts。

```typescript
export function divide(left: number, right: number): number {
  if (right === 0) throw new RangeError("zero divisor");
  return left / right;
}
export async function divideAsync(left: number, right: number): Promise<number> {
  return divide(left, right);
}
```

## 3 类型测试与 @ts-expect-error

先用显式变量类型确认正常结果关系。@ts-expect-error 要求下一行产生诊断；如果该行变成合法代码，会报告 TS2578，使过期的抑制暴露出来。

这个指令只保证下一行有错误，不能证明错误就是实参类型。因此另保留不带抑制的同一调用，在 errors:23 核对 TS2345 和具体文件位置。不要只数错误数量，也不要把意外解析错误当成预期类型错误。

以下片段来自 type-tests.ts。

```typescript
import { divide, divideAsync } from "./calculator.js";
const result: number = divide(9, 3);
const pending: Promise<number> = divideAsync(9, 3);
// @ts-expect-error 字符串不是数值参数；原始错误另由 errors:23 核对。
divide("9", 3);
void result;
void pending;
```

以下片段来自 type-errors.ts。

```typescript
import { divide } from "./calculator.js";
divide("9", 3); // TS2345：字符串参数。
// @ts-expect-error TS2578：这里故意不产生类型错误，用来观察未使用指令。
divide(9, 3);
```

Step 1：读取独立反例的原始诊断。

```bash
npm run errors:23
# 退出 1；TS2345 为字符串参数，TS2578 为多余的预期错误指令。
```

## 4 同步、异步与异常断言

node:test 注册测试，node:assert/strict 提供严格断言。equal 检查标量，deepEqual 检查数组结构，throws 接收稍后调用的函数并检查异常。不要先执行抛错表达式再把结果传给 throws。

rejects 返回 Promise，异步测试必须 await 它。遗漏 await 可能让测试函数先结束，剩余异步活动不能作为已完成的测试成果。这里使用立即完成或拒绝的 Promise，不靠计时延迟等待结果。

以下片段来自 calculator.test.ts。

```typescript
import test from "node:test";
import assert from "node:assert/strict";
import { divide, divideAsync } from "./calculator.js";
test("同步值和异常", () => {
  assert.equal(divide(9, 3), 3);
  assert.equal(divide(0, 3), 0);
  assert.throws(() => divide(1, 0), { name: "RangeError", message: "zero divisor" });
});
test("异步完成和拒绝", async () => {
  assert.equal(await divideAsync(9, 3), 3);
  await assert.rejects(divideAsync(1, 0), { name: "RangeError", message: "zero divisor" });
});
test("每个测试自己创建可变输入", () => {
  const values = [9, 3];
  assert.deepEqual(values.map(value => divide(value, 3)), [3, 1]);
});
```

## 5 持续集成与测试隔离

持续集成按固定版本安装、类型检查、构建、运行测试的顺序进行。仓库已有 npm 锁文件时使用相同环境入口；不能让构建失败后的旧 JavaScript 继续充当本次测试产物。

每个测试自己创建可变数据。Node 测试运行器通常按文件隔离进程，同一文件内仍可能共享模块状态；并发选项和测试顺序不能弥补污染。涉及资源时应在测试清理钩子或 finally 中结束并释放。

类型检查约束可赋值关系，代码规范检查约束选定规则，运行测试观察本次输入的行为，三者不能互相替代。本例不安装代码规范工具；TypeScript 7.0 没有旧的编译器 API，需依赖该 API 的工具必须先核对兼容性，不直接假定升级 tsc 后它们也能工作。

## 本章小结

- 正常配置和预期错误配置分开，检查文件范围比单个表达式是否报错更重要。
- @ts-expect-error 配合原始诊断才能确认错误原因。
- 异步测试需要等待完成；环境和输入隔离使测试能重复执行。

## 练习

1. 增加负数除法测试，检查 divide(-9, 3) 为 -3，零除数仍抛同一异常。
2. 暂时把原始字符串反例改为数值，确认 TS2345 消失；再观察类型测试中多余的 @ts-expect-error 如何被检测。
3. 为 divideAsync(0, 3) 写异步测试，必须 await，并核对结果为 0、测试总数增加、fail 仍为 0。

## 参考与引用来源

- TypeScript 官方文档：[3.9 / @ts-expect-error](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-3-9.html#-ts-expect-error-comments)、[TSConfig Reference](https://www.typescriptlang.org/tsconfig/) 中 files、noEmit、noEmitOnError：检查范围与错误测试。
- Node.js 24.11.0：[测试执行模型](https://nodejs.org/download/release/v24.11.0/docs/api/test.html#test-runner-execution-model)、[额外异步活动](https://nodejs.org/download/release/v24.11.0/docs/api/test.html#extraneous-asynchronous-activity)、[严格断言](https://nodejs.org/download/release/v24.11.0/docs/api/assert.html#strict-assertion-mode) 与 assert.throws、assert.rejects：同步、异步、异常及隔离。
- Microsoft Developer Blogs：[TypeScript 7.0 / Running Side-by-Side with TypeScript 6.0](https://devblogs.microsoft.com/typescript/announcing-typescript-7-0/#running-side-by-side-with-typescript-6-0)：旧编译器 API 工具的兼容边界。